# Visualize and test EM synaptome

In this notebook we select and download an `EM synaptome`. That is:
  - a skeleton representation of a single neuron from an electron microscopic dense tissue reconstruction
  - plus the spines on its surface, individually extracted
  - plus all afferent synapses onto the neuron, mapped to individual spines, shafts or the soma

This notebook serves as a starting point to a new user. It shows ways to access synapse and spine information. It provides a visualization of cell surface meshes, spines and afferent synapses together.


## Platform authentication
We begin by authenticating with the OBI platform to be able to access the synaptomes.

Please follow the instruction below to authenticate, then select the project to work with. 
The synaptome you want to visualize must be either public, or generated within that project.


In [ ]:
import os
import obi_auth
import numpy
import h5py
import tqdm
import bluepysnap as snap
from obi_notebook.get_projects import get_projects
from obi_notebook.get_entities import get_entities
from obi_notebook import get_environment
from entitysdk import Client
from entitysdk.models import CellMorphology, Subject, EMCellMesh, Circuit
from entitysdk.staging import stage_circuit
from morph_spines import load_morphology_with_spines

from trimesh import util as triutil
from trimesh.visual.color import ColorVisuals
from trimesh.exchange import obj
from trimesh import Trimesh

from pathlib import Path

environment = get_environment.get_environment()
token = obi_auth.get_token(environment=environment, auth_mode="daf")
project_context = get_projects(token)

## Selecting a synaptome

**IMPORTANT. Read this carefully** 

`Synaptomes` is what we call simulatable models of a neuron and its afferents. There are many on the OBI platform, but not all of them have been generated from electron microscopy. Others have been built as statistical models by stochastic algorithms.

For this notebook, we require a `Synaptome` from electron microscopy. Below, you will be provided with a table to select a `Synaptome` from. Please make sure you select one where the value in the last column is `em_reconstruction`. Otherwise, in the next cell an error will be raised.


In [ ]:
client = Client(environment=environment, token_manager=token, project_context=project_context)
circ_ids = []
circ_ids = get_entities(entity_type="circuit", token=token, result=circ_ids, project_context=project_context,
                       exclude_scales=["pair", "small", "microcircuit", "region", "system", "whole_brain"],
                       add_columns=["build_category"], page_size=50)
                       


## Download the `Synaptome`. Load neuron morphology and synapses


In [ ]:
circ_entity = client.get_entity(entity_id=circ_ids[0], entity_type=Circuit)
if circ_entity.build_category != "em_reconstruction":
    raise ValueError("The selected synaptome is not derived from an electron microscopic reconstruction!")

# This downloads the selected Synaptome to the local system.
circ_path = stage_circuit(client=client, model=circ_entity, output_dir=Path("downloaded_synaptome"))

# This loads the Synaptome as a `Circuit` object
circ = snap.Circuit(circ_path)
# Technically, the single neuron is represented as a `node population`. We access it.
node = circ.nodes["biophysical_neuron"]
# All the presynaptic neurons innervating it are a second `node population`. We get their types.
afferents = circ.nodes["synaptome_afferent_neurons"].get(properties="synapse_class")

# This loads the neuron along with the detected spines. Later we will see how to access spines.
m = load_morphology_with_spines(node.config["alternate_morphologies"]["h5v1"])

# Technically, the synapses are represented as an `edge population`. We access it here.
edge = circ.edges['synaptome_afferents']
# We load the synapses of the synaptome neuron. First argument is an identifier of the neuron.
# Since a Synaptome contains only a single biophysical neuron, we just fill in 0.
syns = edge.afferent_edges(0, properties=edge.property_names)
syns["presyn_type"] = afferents[syns["@source_node"]].to_numpy()


## Download the source cell surface mesh

We also want to load the cell surface mesh that the morphology with spines was created from. 
We then visualize them together. This will allow you to assess the accuracy of the created morphology and its spines.

We download the mesh by finding the associated `EMCellMesh` in our database. We find it by matching its `pt_root_id`, which is a unique identifier of an EM neuron.


In [ ]:
pt_root_id = int(circ_entity.name.split("-")[-1])
microns_mesh = list(client.search_entity(entity_type=EMCellMesh, query={
    "dense_reconstruction_cell_id": pt_root_id
}))[0]

root = "morphology_meshes"
os.makedirs(root, exist_ok=True)
path_dl_mesh = client.download_file(entity_id=microns_mesh.id,
                                    entity_type=EMCellMesh,
                                    asset_id=microns_mesh.assets[0].id,
                                    output_path=root)

### Create aggregate mesh of all spines.

Here, we iterate over all spines, load their individual meshes and aggregate them into a single mesh that can be used and visualized.

Uncomment the last line of the cell to show all spine meshes in situ, but beware: It may take a while, to show as there are probably thousands of them.

In [ ]:
colors = [
    [255, 40, 40]
]
n = len(colors) + 1

splt = numpy.linspace(0, m.spines.spine_count, n).astype(int)
rnd = numpy.random.permutation(m.spines.spine_count)
grps = [
    rnd[a:b] for a, b in zip(splt[:-1], splt[1:])
]


spine_grp_meshes = []

for grp, col in zip(grps, colors):
    mesh = triutil.concatenate(
        [m.spines.spine_mesh(_i) for _i in tqdm.tqdm(grp)]
    )
    mesh.visual = ColorVisuals(mesh=mesh, face_colors=col)
    spine_grp_meshes.append(mesh)

spine_grp_meshes = triutil.concatenate(
    spine_grp_meshes
)

# spine_grp_meshes.show()

### Create a mesh of the neuron without spines

Now we load the source cell surface mesh. That is, a representation of the neuron surface including spines.

Then, we iterate over the detected spines and remove them from the mesh one by one. The result is a mesh of only the neurites and soma, but where spines used to be only a hole remains. 

Uncomment the last line to show that mesh.

In [ ]:
with open(path_dl_mesh, "r") as fid:
    all_mesh = obj.load_obj(fid)

all_mesh = all_mesh["geometry"][str(path_dl_mesh)]
all_mesh["vertices"] = all_mesh["vertices"] / 1000.0
all_mesh = Trimesh(**(all_mesh))
all_mesh.visual = ColorVisuals(mesh=all_mesh, face_colors=[210, 210, 210])

tst = all_mesh.kdtree.query_ball_tree(spine_grp_meshes.kdtree, 8E-2)
_v = numpy.array(list(map(len, tst))) == 0
nz = numpy.nonzero(_v)[0]
all_mesh.update_faces(numpy.all(numpy.isin(all_mesh.faces, nz), axis=1))
all_mesh.update_vertices(all_mesh.referenced_vertices)

# all_mesh.show()

## Visualize morphology skeleton, spines -- and synapses

Now we visualize everything together: The mesh of the neurites without spines, individual spines and the synapses mapped onto them.
The synapses are colored in accordance with the spine they are mapped to, or grey for shaft synapses.

### Define plotting helper functions
First we define helper functions

In [ ]:
import trimesh
from trimesh import bounds
from copy import deepcopy
from scipy.spatial import KDTree
from trimesh.visual.color import hsv_to_rgba

def apply_bounding_box_to_mesh(bbox, mesh):
    mesh = deepcopy(mesh)
    nz = numpy.nonzero(bounds.contains(bbox, mesh.vertices))[0]
    mesh.update_faces(numpy.all(numpy.isin(mesh.faces, nz), axis=1))
    mesh.update_vertices(mesh.referenced_vertices)
    return mesh

def apply_filter_to_mesh(func, mesh):
    mesh = deepcopy(mesh)
    nz = numpy.nonzero(func(mesh.vertices))[0]
    mesh.update_faces(numpy.all(numpy.isin(mesh.faces, nz), axis=1))
    mesh.update_vertices(mesh.referenced_vertices)
    return mesh

def filter_close_to_section_points(section_id):
    sec_pts = m.morphology.section(section_id).points[:, :3]
    def func(pts):
        tree = KDTree(sec_pts)
        return [len(_x) > 0 for _x in tree.query_ball_point(pts, 1.5)]
    return func

def cyclic_rgb_for_spines(sec_pos):
    n_hues = 4
    hues = numpy.linspace(0, 1.0, n_hues + 1)[:-1]
    hsv = numpy.vstack([hues, numpy.ones(len(hues)), 0.99 * numpy.ones(len(hues))]).transpose()
    idxx = sec_pos.argsort().to_numpy()
    hsv_out = -numpy.ones((len(idxx), hsv.shape[1]))
    hsv_out[idxx] = hsv[numpy.mod(numpy.arange(len(idxx)), n_hues)]

    return dict(zip(sec_pos.index, hsv_to_rgba(hsv_out)))

def meshes_for_spines_on(section_id):
    spine_ids = m.spines.spine_table.index[m.spines.spine_table["afferent_section_id"] == section_id]
    rgbs = cyclic_rgb_for_spines(m.spines.spine_table.loc[spine_ids, "afferent_section_pos"])

    spines_mesh = []
    for _i in spine_ids:
        _col = rgbs[_i]
        _mesh = m.spines.spine_mesh(_i)
        _mesh.visual = ColorVisuals(mesh=_mesh, face_colors=_col)
        spines_mesh.append(_mesh)
    spines_mesh = triutil.concatenate(spines_mesh)
    return spines_mesh, rgbs

types_to_color = {
    "extrinsic_neuron": [0, 255, 0, 255],
    "excitatory_neuron": [255, 0, 0, 255],
    "inhibitory_neuron": [0, 0, 255, 255]
}

def synapse_meshes_on(section_id, rgbs, radius=0.35, color_by_spine=True):
    xyz = ["afferent_synapse_x", "afferent_synapse_y", "afferent_synapse_z"]
    syn_spheres = syns.loc[syns.afferent_section_id == section_id,
                           xyz + ["spine_sharing_id", "presyn_type"]]
    grey = numpy.array([210, 210, 210, 255])
    def make_sphere(row):
        sphere = trimesh.primitives.Sphere(radius=radius, center=row[xyz].to_numpy())
        if color_by_spine:
            sphere_col = rgbs.get(row["spine_sharing_id"], grey).copy()
        else:
            sphere_col = types_to_color[row["presyn_type"]].copy()
        sphere_col[-1] = 0.3
        sphere.visual = ColorVisuals(mesh=sphere, face_colors=sphere_col)
        return sphere
    print(syn_spheres["presyn_type"].value_counts())
    syn_spheres = syn_spheres.apply(make_sphere, axis=1)
    return triutil.concatenate(syn_spheres)



## Select morphology section to show.

Showing all the information for the entire morphology at once would probably be too much. 

So, here we select a morphology section to show. It will then be plotted in the next cell.

Synapses that were detected in the EM volume are shown as 350 nm spheres. By default they are colored the same as the spine they are mapped to, or grey if they are on a shaft. This way you can assess the mapping of synapses to morphology locations.

You can also select an alternative color scheme where they are colored by the class of the afferent neuron: EXC or INH. However, note that most synapses are extrinsic and thus no information about the afferent neuron is known. Also, coloring by a more fine grained scheme (different INH subtypes) could be easily implemented!

In [ ]:
from ipywidgets import widgets

section_spine_counts = m.spines.spine_table["afferent_section_id"].value_counts()
labels = [f"Section {idx}: {n} spines" for idx, n in
          zip(section_spine_counts.index, section_spine_counts.to_numpy())]
selector = widgets.Dropdown(options=list(zip(labels, section_spine_counts.index)))
color_scheme = widgets.Dropdown(options=[("By spine", True), ("EXC:red, INH:blue, extrinsic:green", False)])
display(selector, color_scheme)

In [ ]:
def show_for_section_id(section_id):
    spine_meshes, rgbs = meshes_for_spines_on(section_id)
    show_mesh = triutil.concatenate([
        apply_filter_to_mesh(filter_close_to_section_points(section_id - 1), all_mesh),
        spine_meshes,
        synapse_meshes_on(section_id, rgbs, radius=0.35, color_by_spine=color_scheme.value)
    ])
    return show_mesh.show()

show_for_section_id(selector.value)